# UC Admissions Data Challenge — Question Sprint

Upload these files to Colab before running: `bay_area_modeling_table.csv`, `dashboard_data.csv`, `uc_admissions_summary_by_ethnicity.csv`, and `uc_freshman_admission_by_discipline.csv`.

In [ ]:
import pandas as pd

bay = pd.read_csv('/content/bay_area_modeling_table.csv', low_memory=False)
dashboard = pd.read_csv('/content/dashboard_data.csv', low_memory=False)
eth = pd.read_csv('/content/uc_admissions_summary_by_ethnicity.csv')
discipline = pd.read_csv('/content/uc_freshman_admission_by_discipline.csv')

def rate(frame, numerator='admits', denominator='applicants'):
    return frame[numerator].sum() / frame[denominator].sum()


## Question 1 — Average number of UC campuses applied to

In [ ]:
q1_campus = bay.query("fall_term == 2025 and campus != 'Universitywide'")
q1_systemwide = bay.query("fall_term == 2025 and campus == 'Universitywide'")
q1 = q1_campus['applicants'].sum() / q1_systemwide['applicants'].sum()
print(f'{q1:.2f}')

## Question 2 — Fall 2025 UCLA admit rate

In [ ]:
q2_data = bay.query("fall_term == 2025 and campus == 'Los Angeles'")
q2 = rate(q2_data)
print(f'{q2:.3f}')

## Question 3 — Campus with the largest Computer Science admit-rate penalty

In [ ]:
overall = discipline.query("broad_discipline == 'All disciplines'")[["campus", "admit_rate"]].rename(columns={"admit_rate": "overall_rate"})
cs = discipline.query("broad_discipline == 'Computer Science'")[["campus", "admit_rate"]].rename(columns={"admit_rate": "cs_rate"})
q3_table = overall.merge(cs, on='campus')
q3_table['penalty'] = q3_table['overall_rate'] - q3_table['cs_rate']
print(q3_table.loc[q3_table['penalty'].idxmax(), 'campus'])
q3_table.sort_values('penalty', ascending=False)

## Question 4 — Berkeley Computer Science admit-GPA interquartile range

In [ ]:
q4_row = discipline.query("campus == 'Berkeley' and broad_discipline == 'Computer Science'").iloc[0]
q4 = q4_row['admit_gpa_p75'] - q4_row['admit_gpa_p25']
print(f'{q4:.3f}')

## Question 5 — Campuses where White admit rate exceeds Hispanic/Latino(a) admit rate

In [ ]:
q5 = eth.query("entrant_level == 'freshman' and fall_term == 2025")
q5_rates = q5.pivot_table(index='campus', columns='ethnicity', values='n', aggfunc='sum')
q5_rates['white_rate'] = q5_rates[('Adm', 'White')] / q5_rates[('App', 'White')] if isinstance(q5_rates.columns, pd.MultiIndex) else None

adm = q5.query("count_type == 'Adm'").pivot(index='campus', columns='ethnicity', values='n')
app = q5.query("count_type == 'App'").pivot(index='campus', columns='ethnicity', values='n')
q5_rates = pd.DataFrame({
    'White': adm['White'] / app['White'],
    'Hispanic/Latino(a)': adm['Hispanic/Latino(a)'] / app['Hispanic/Latino(a)']
})
q5_rates['white_higher'] = q5_rates['White'] > q5_rates['Hispanic/Latino(a)']
print(int(q5_rates['white_higher'].sum()))
q5_rates

## Question 6 — Higher systemwide freshman admit rate

In [ ]:
q6 = eth.query("entrant_level == 'freshman' and fall_term == 2025 and campus == 'Systemwide'")
q6_rates = q6.pivot(index='ethnicity', columns='count_type', values='n')
q6_rates['admit_rate'] = q6_rates['Adm'] / q6_rates['App']
print(q6_rates['admit_rate'].idxmax())
q6_rates

## Question 7 — Bay Area graduates enrolling at a California Community College

In [ ]:
q7_data = bay.query("fall_term == 2023 and campus == 'Universitywide'")
q7 = q7_data['enrolled_ccc'].sum() / q7_data['hs_completers'].sum()
print(f'{q7:.3f}')

## Question 8 — Mission San Jose UC applicants divided by a-g completers

In [ ]:
q8_row = bay.query("fall_term == 2023 and campus == 'Universitywide' and high_school == 'MISSION SAN JOSE HIGH SCHOOL'").iloc[0]
q8 = q8_row['applicants'] / q8_row['ag_completers']
print(f'{q8:.3f}')

## Question 9 — Distinct public high schools with at least one freshman applicant

In [ ]:
q9_data = bay.query("fall_term == 2025 and applicants > 0 and ~school_type.fillna('').str.contains('private', case=False)")
q9 = q9_data['atp_code'].nunique()
print(q9)

## Question 10 — School that most outperforms expected Berkeley admit rate

In [ ]:
schools = [
    'HERCULES HIGH SCHOOL',
    'MISSION SENIOR HIGH SCHOOL',
    'MONTEREY TRAIL HIGH SCHOOL',
    'PHILLIP & SALA BURTON ACAD HS',
    'RANCHO SAN JUAN HIGH SCHOOL'
]
q10_data = dashboard.query("campus == 'Berkeley' and fall_term.between(2022, 2025)").copy()
q10_data = q10_data[q10_data['high_school'].isin(schools)]
q10_data['residual'] = q10_data['admit_rate'] - q10_data['expected_admit_rate']
q10_summary = q10_data.groupby('high_school')['residual'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(q10_summary.index[0])
q10_summary